# 100-Task Failure-Intensive HumanEval Steering Benchmark Suite (`Qwen/Qwen3-4B`)
### Optimized for NVIDIA Tesla T4 | Two-Phase Failure-Recovery Steering Comparison

---

### Two-Phase Protocol
1. **Phase A: Vanilla screen.** The greedy vanilla baseline runs on all 100 curated HumanEval tasks. Every task it fails is carried to Phase B.
   Runs in chunks of `CHUNK_SIZE = 20` tasks: re-run the cell until it reports completion (checkpointed, so a Colab disconnect loses at most one task).
2. **Phase B: Failure recovery.** Only the vanilla failures are re-run with the three interventions below, and we measure how many each one recovers. Also chunked by 20 tasks.

### Methods
1. **Vanilla Baseline** ($T=0.0$ greedy decoding, no guardrails or hooks). Phase A, all 100 tasks.
2. **Negative System Prompting** (instructional guardrail forbidding syntax, scoping and logic bugs)
3. **Adaptive RepGated Steering** ($\alpha=0.08$, $\tau = 2/\sqrt{d_{\text{model}}} \approx 0.04$, Layer 18 gated activation hook repelling $v_{\text{fail}}$)
4. **Dual Push-Pull Qwen Trace Steering** ($\alpha=0.08, \beta=0.015$, Layer 18 failure push + Qwen3 reasoning-trace pull during $t \le 200$) ⭐

Static failure steering is not part of this comparison.

### Evaluation Harness Fixes
* **Tests actually run.** HumanEval's `test` field only *defines* `check(candidate)`. The earlier harness never called it, so any code that executed counted as a pass. The harness now appends `check(entry_point)`.
* **Sandboxed with a timeout.** Each program runs in a subprocess (10 s limit, stdin closed), so infinite loops or `input()` calls cannot hang the notebook.
* **Qwen3 thinking mode is off by default.** Qwen advises against greedy decoding in thinking mode (endless repetition), and at 1024 tokens the reasoning was being truncated before any code was written. Set `ENABLE_THINKING = True` and raise `TOKEN_BUDGET` to study thinking mode.
* **Cleaner code extraction.** Helper functions, imports and constants are kept; top-level demo code (`print(...)`, `if __name__ == "__main__":`) is dropped.

### Metrics
* **Recovery rate** on vanilla failures with **95% Wilson CIs**
* **Exact McNemar test** (binomial on discordant pairs) vs. vanilla
* **Projected overall Pass@1** (upper bound: vanilla passes + recovered tasks)
* **Shannon entropy** $H(X)$, **3-gram repetition** $R_{\text{rep}}$, **throughput**, **truncation rate**
* **Error taxonomy** (`SUCCESS`, `AssertionError`, `SyntaxError`, `NameError`, `TypeError`, `IndexError`, `Timeout`, ...)
* **Per-task recovery heatmap** and **trajectory comparator** (`plot_task_trajectory_comparison`)

## ⚡ Section 1: Tesla T4 & GPU Acceleration Setup

In [ ]:
# Install required dependencies
# Only upgrade transformers (Qwen3 needs >=4.51); leave Colab's preinstalled torch/numpy/pyarrow alone
!pip install -q "transformers>=4.51.0" accelerate datasets

import torch
import os
import sys
import gc

# Enable Tensor Core TF32 acceleration (Ampere+) & cuDNN autotuner for all GPUs
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Model: {gpu_name}")
    print(f"VRAM Total: {vram_gb:.2f} GB")
    # Check for Turing (Tesla T4) vs Ampere+ architecture
    is_turing = "T4" in gpu_name or "2080" in gpu_name or "TU" in gpu_name
    optimal_dtype = torch.float16 if is_turing or not torch.cuda.is_bf16_supported() else torch.bfloat16
    print(f"Optimal Tensor Precision Selected: {optimal_dtype} ({'Tesla T4 FP16 Tensor Core Hardware' if is_turing else 'Ampere/Ada Native BF16'})")
else:
    optimal_dtype = torch.float32
    print("⚠️ Running on CPU mode. GPU runtime strongly recommended.")

## 📦 Section 2: Core Steering & Continuous Quality Tracking Modules

In [ ]:
import re
import math
import time
import io
import contextlib
import numpy as np
import torch
import torch.nn as nn
from typing import List, Tuple, Dict, Any, Optional, Union
from scipy import stats
from sklearn.svm import SVC

# Qwen3 hybrid reasoning switch. Qwen advises against greedy decoding in thinking mode (endless repetition),
# and greedy thinking routinely exhausted the 1024-token budget before writing any code.
# False = direct answers (default). True = <think>...</think> reasoning; raise TOKEN_BUDGET (>= 4096) if enabled.
ENABLE_THINKING = False

def fix_seeds(seed: int = 42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

def format_chatml_prompt(tokenizer: Any, prompt: str) -> str:
    if "<|im_start|>" in prompt or "<|user|>" in prompt or "<think>" in prompt:
        return prompt
    system_prompt = (
        "You are an expert Python programmer. Solve the user's coding request directly by writing a clean, "
        "valid, and executable Python function. "
        + ("Keep reasoning brief inside <think>...</think>, and output " if ENABLE_THINKING else "Output ")
        + "the requested Python function inside a ```python ``` block."
    )
    user_content = (
        "Complete the following Python function code. "
        "Provide a complete, executable Python function definition starting with `def ...` inside a ```python ... ``` block.\n\n"
        f"{prompt}"
    )
    if hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None) is not None:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
        except Exception:
            pass
    return f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n"

def format_negative_system_prompt(tokenizer: Any, prompt: str) -> str:
    if "<|im_start|>" in prompt or "<|user|>" in prompt or "<think>" in prompt:
        return prompt
    system_prompt = (
        "You are an expert Python programmer. Solve the user's coding request directly by writing a clean, "
        "valid, and executable Python function.\n"
        "STRICT GUARDRAILS:\n"
        "1. Do NOT output unindented code outside the function definition.\n"
        "2. Do NOT leave variables uninitialized or reference undefined names.\n"
        "3. Do NOT produce IndexError, TypeError, or SyntaxError.\n"
        "4. Output ONLY the requested Python function inside a ```python ``` block."
    )
    user_content = f"Complete the following Python function code:\n\n{prompt}"
    if hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None) is not None:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
        except Exception:
            pass
    return f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n"

CONTRASTIVE_PAIRS = [
    {
        "buggy": "def add_elements(arr):\n    res = []\n    for i in range(len(arr) + 1):\n        res.append(arr[i])\n    return res",
        "correct": "def add_elements(arr):\n    res = []\n    for i in range(len(arr)):\n        res.append(arr[i])\n    return res"
    },
    {
        "buggy": "def concatenate_strings(s1, s2):\n    return s1 + 10",
        "correct": "def concatenate_strings(s1, s2):\n    return s1 + str(10)"
    },
    {
        "buggy": "def compute_total(numbers):\n    for num in numbers:\n        total += num\n    return total",
        "correct": "def compute_total(numbers):\n    total = 0\n    for num in numbers:\n        total += num\n    return total"
    },
    {
        "buggy": "def parse_data(items):\n    for item in items\n        print(item)",
        "correct": "def parse_data(items):\n    for item in items:\n        print(item)"
    },
    {
        "buggy": "def check_palindrome(s):\n    res = s.lower()\n    return res",
        "correct": "def check_palindrome(s):\n    res = s.lower()\n    return (res, res == res[::-1])"
    },
    {
        "buggy": "def compute_fib(n):\n    return compute_fib(n-1) + compute_fib(n-2)",
        "correct": "def compute_fib(n):\n    if n <= 1:\n        return n\n    return compute_fib(n-1) + compute_fib(n-2)"
    },
    {
        "buggy": "def mod_exp(x, p):\n    return (x ** 2) % p + 1",
        "correct": "def mod_exp(x, p):\n    return (x ** 2) % p"
    },
    {
        "buggy": "def get_value(d, key):\n    return d[key]",
        "correct": "def get_value(d, key):\n    return d.get(key, None)"
    }
]

# Reasoning traces written in Qwen3's own thinking-mode style ("Okay, let's see..." + self-checks,
# blank line after </think>, matching the Qwen3 chat template). Used to extract the v_gist pull vector.
QWEN_REASONING_TRACES = [
    "<think>\nOkay, let's see. I need to write a function that processes the input sequence.\nFirst, I should handle the edge cases: an empty list or a single element.\nThen I'll initialize the accumulator variables before the loop so nothing is undefined.\nWait, the loop has to stay inside range(len(arr)), not len(arr) + 1, or I'll get an IndexError.\nLet me double-check the docstring examples... yes, the return type should match exactly.\n</think>\n\n```python\ndef solution():\n    pass\n```",
    "<think>\nAlright, the task is to parse nested parentheses.\nHmm, I can track the current depth with a counter and keep a separate max_depth variable.\nFor each character, increment on '(' and decrement on ')', updating max_depth as I go.\nI need to iterate over the characters directly so there is no index overflow.\nLet me verify with the example '(()())': depth goes 1, 2, 1, 2, 1, 0, so the max is 2. That's correct.\n</think>\n\n```python\ndef solution():\n    pass\n```"
]

class MaximumMarginVectorExtractor:
    def __init__(self, model: Optional[Any] = None, tokenizer: Optional[Any] = None, model_name: str = "Qwen/Qwen3-4B", device: Optional[str] = None):
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        fix_seeds(42)
        if model is not None and tokenizer is not None:
            self.model = model
            self.tokenizer = tokenizer
        else:
            from transformers import AutoTokenizer, AutoModelForCausalLM
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
            is_turing = "T4" in gpu_name or "2080" in gpu_name or "TU" in gpu_name
            dtype_to_use = torch.float16 if (is_turing or not torch.cuda.is_bf16_supported()) else torch.bfloat16

            load_kwargs = {
                "torch_dtype": dtype_to_use,
                "device_map": "auto" if self.device == "cuda" else None,
                "trust_remote_code": True
            }
            try:
                self.model = AutoModelForCausalLM.from_pretrained(model_name, attn_implementation="sdpa", **load_kwargs)
            except Exception:
                self.model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
            self.model.eval()

    def get_layer_activations(self, prompts: List[str], target_layer: int = 18) -> torch.Tensor:
        activations = []
        def hook_fn(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            activations.append(hidden[:, -1, :].detach().cpu().to(torch.float32))

        layer = self.model.model.layers[target_layer] if hasattr(self.model, "model") else self.model.transformer.h[target_layer]
        handle = layer.register_forward_hook(hook_fn)
        try:
            with torch.inference_mode():
                for p in prompts:
                    formatted = format_chatml_prompt(self.tokenizer, p)
                    inputs = self.tokenizer(formatted, return_tensors="pt").to(self.model.device)
                    self.model(**inputs)
        finally:
            handle.remove()
        return torch.cat(activations, dim=0)

    def fit_maximum_margin_vector(self, buggy_prompts: List[str], correct_prompts: List[str], target_layer: int = 18) -> torch.Tensor:
        act_buggy = self.get_layer_activations(buggy_prompts, target_layer=target_layer)
        act_correct = self.get_layer_activations(correct_prompts, target_layer=target_layer)
        X = torch.cat([act_buggy, act_correct], dim=0).numpy()
        y = np.array([1] * len(act_buggy) + [-1] * len(act_correct))
        clf = SVC(kernel="linear", C=1.0)
        clf.fit(X, y)
        w = torch.tensor(clf.coef_[0], dtype=torch.float32)
        w_unit = w / torch.norm(w, p=2)
        return w_unit.to(self.model.device, dtype=self.model.dtype)

class QwenGistVectorExtractor:
    def __init__(self, model: Any, tokenizer: Any, device: Optional[str] = None):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")

    def extract_gist_vector(self, target_layer: int = 18, v_fail: Optional[torch.Tensor] = None) -> torch.Tensor:
        activations = []
        def hook_fn(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            activations.append(hidden[:, -1, :].detach().cpu().to(torch.float32))

        layer = self.model.model.layers[target_layer] if hasattr(self.model, "model") else self.model.transformer.h[target_layer]
        handle = layer.register_forward_hook(hook_fn)
        try:
            with torch.inference_mode():
                for trace in QWEN_REASONING_TRACES:
                    inputs = self.tokenizer(trace, return_tensors="pt").to(self.model.device)
                    self.model(**inputs)
        finally:
            handle.remove()

        v_gist_raw = torch.cat(activations, dim=0).mean(dim=0)
        if v_fail is not None:
            v_fail_f32 = v_fail.detach().cpu().to(torch.float32)
            proj = (torch.dot(v_gist_raw, v_fail_f32) / torch.dot(v_fail_f32, v_fail_f32)) * v_fail_f32
            v_gist_raw = v_gist_raw - proj
        v_gist_unit = v_gist_raw / torch.norm(v_gist_raw, p=2)
        return v_gist_unit.to(self.model.device, dtype=self.model.dtype)

class AdaptiveGatedSteeredGenerator:
    def __init__(self, model: Any, tokenizer: Any):
        self.model = model
        self.tokenizer = tokenizer

    def generate_gated(self, prompt: str, v_fail: torch.Tensor, target_layer: int = 18, alpha: float = 0.08, tau: float = 0.04, max_new_tokens: int = 1024) -> Tuple[str, Dict[str, Any]]:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        prompt_len = inputs.input_ids.shape[1]
        v_steer = -alpha * (v_fail / torch.norm(v_fail, p=2))
        gated_history = []
        s_fail_history = []

        def gating_hook(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            h_last = hidden[:, -1, :].clone()
            norm_h = torch.norm(h_last, p=2, dim=-1, keepdim=True) + 1e-8
            s_val = torch.sum((h_last / norm_h) * v_fail, dim=-1).item()
            s_fail_history.append(float(s_val))
            if s_val > tau:
                hidden[:, -1, :] = hidden[:, -1, :] + v_steer
                gated_history.append(1)
            else:
                gated_history.append(0)
            return (hidden,) + output[1:] if isinstance(output, tuple) else hidden

        layer = self.model.model.layers[target_layer] if hasattr(self.model, "model") else self.model.transformer.h[target_layer]
        handle = layer.register_forward_hook(gating_hook)
        try:
            with torch.inference_mode():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    use_cache=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )
        finally:
            handle.remove()

        gen_tokens = out[0][prompt_len:]
        text = self.tokenizer.decode(gen_tokens, skip_special_tokens=True)
        gating_ratio = (sum(gated_history) / max(len(gated_history), 1)) * 100.0
        stats = {
            "total_tokens": len(gen_tokens),
            "gating_ratio": gating_ratio,
            "s_fail_history": s_fail_history
        }
        return text, stats

class DualPushPullSteeredGenerator:
    def __init__(self, model: Any, tokenizer: Any):
        self.model = model
        self.tokenizer = tokenizer

    def generate_dual_steered(self, prompt: str, v_fail: torch.Tensor, v_gist: torch.Tensor, target_layer: int = 18, alpha: float = 0.08, beta: float = 0.015, cot_warmup_tokens: int = 200, max_new_tokens: int = 1024) -> Tuple[str, Dict[str, Any]]:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        prompt_len = inputs.input_ids.shape[1]
        v_push = -alpha * (v_fail / torch.norm(v_fail, p=2))
        v_pull = beta * (v_gist / torch.norm(v_gist, p=2))
        step_counter = [0]
        s_fail_history = []
        s_gist_history = []

        def dual_steering_hook(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            t_curr = step_counter[0]
            h_last = hidden[:, -1, :].clone()
            norm_h = torch.norm(h_last, p=2, dim=-1, keepdim=True) + 1e-8
            s_fail = torch.sum((h_last / norm_h) * v_fail, dim=-1).item()
            s_gist = torch.sum((h_last / norm_h) * v_gist, dim=-1).item()
            s_fail_history.append(float(s_fail))
            s_gist_history.append(float(s_gist))

            # Apply constant failure repulsion push
            delta = v_push
            # Apply reasoning CoT attraction pull during early generation
            if t_curr <= cot_warmup_tokens:
                delta = delta + v_pull
            hidden[:, -1, :] = hidden[:, -1, :] + delta
            step_counter[0] += 1
            return (hidden,) + output[1:] if isinstance(output, tuple) else hidden

        layer = self.model.model.layers[target_layer] if hasattr(self.model, "model") else self.model.transformer.h[target_layer]
        handle = layer.register_forward_hook(dual_steering_hook)
        try:
            with torch.inference_mode():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    use_cache=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )
        finally:
            handle.remove()

        gen_tokens = out[0][prompt_len:]
        text = self.tokenizer.decode(gen_tokens, skip_special_tokens=True)
        stats = {
            "total_tokens": len(gen_tokens),
            "s_fail_history": s_fail_history,
            "s_gist_history": s_gist_history
        }
        return text, stats

class EntropyEvaluator:
    def __init__(self, model: Any, tokenizer: Any):
        self.model = model
        self.tokenizer = tokenizer

    def compute_token_diagnostics(self, prompt: str, generated_text: str) -> Dict[str, Any]:
        full_text = prompt + generated_text
        inputs = self.tokenizer(full_text, return_tensors="pt").to(self.model.device)
        prompt_len = self.tokenizer(prompt, return_tensors="pt").input_ids.shape[1]
        with torch.inference_mode():
            outputs = self.model(**inputs)
            logits = outputs.logits[:, prompt_len - 1:-1, :].to(torch.float32)

        if logits.shape[1] == 0:
            return {"mean_entropy": 0.0, "r_rep": 0.0, "entropy_trajectory": []}

        probs = torch.softmax(logits, dim=-1)
        log_probs = torch.log2(torch.clamp(probs, min=1e-12))

        # Tensor-level negation before converting to list:
        entropy_tensor = -torch.sum(probs * log_probs, dim=-1)
        entropies = entropy_tensor.squeeze(0).cpu().tolist()
        mean_entropy = float(np.mean(entropies)) if len(entropies) > 0 else 0.0

        gen_ids = inputs.input_ids[0, prompt_len:].cpu().tolist()
        if len(gen_ids) < 3:
            r_rep = 0.0
        else:
            ngrams = [tuple(gen_ids[i:i+3]) for i in range(len(gen_ids) - 2)]
            unique_ngrams = len(set(ngrams))
            r_rep = 1.0 - (unique_ngrams / max(len(ngrams), 1))

        return {
            "mean_entropy": mean_entropy,
            "r_rep": float(r_rep),
            "entropy_trajectory": entropies
        }

## 🤖 Section 3: Load Qwen/Qwen3-4B Model & Extract Contrastive Vectors ($v_{\text{fail}}$ & $v_{\text{gist}}$)

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B"
TARGET_LAYER = 18

print(f"Loading {MODEL_NAME} for Single-Layer (Layer {TARGET_LAYER}) Failure-Recovery Steering Benchmark...")
extractor_vfail = MaximumMarginVectorExtractor(model_name=MODEL_NAME)
model = extractor_vfail.model
tokenizer = extractor_vfail.tokenizer

NUM_LAYERS = model.config.num_hidden_layers
D_MODEL = model.config.hidden_size
assert TARGET_LAYER < NUM_LAYERS, f"TARGET_LAYER={TARGET_LAYER} out of range for {NUM_LAYERS}-layer model"
print(f"Architecture: {NUM_LAYERS} decoder layers, d_model={D_MODEL} | Thinking mode: {ENABLE_THINKING}")

# Fit failure vector v_fail from 8 contrastive pairs at Layer 18
buggy_prompts = [p["buggy"] for p in CONTRASTIVE_PAIRS]
correct_prompts = [p["correct"] for p in CONTRASTIVE_PAIRS]
v_fail = extractor_vfail.fit_maximum_margin_vector(buggy_prompts, correct_prompts, target_layer=TARGET_LAYER)

# Extract Qwen gist vector v_gist (orthogonalized perp v_fail)
extractor_vgist = QwenGistVectorExtractor(model=model, tokenizer=tokenizer)
v_gist = extractor_vgist.extract_gist_vector(target_layer=TARGET_LAYER, v_fail=v_fail)

# Initialize Steering Generators & Evaluators
generator_gated = AdaptiveGatedSteeredGenerator(model=model, tokenizer=tokenizer)
generator_dual = DualPushPullSteeredGenerator(model=model, tokenizer=tokenizer)
entropy_evaluator = EntropyEvaluator(model=model, tokenizer=tokenizer)

print(f"✅ Vectors successfully initialized at Layer {TARGET_LAYER}:")
print(f"  • v_fail norm: {torch.norm(v_fail, p=2):.4f}")
print(f"  • v_gist norm: {torch.norm(v_gist, p=2):.4f}")
print(f"  • Orthogonality dot product <v_fail, v_gist>: {torch.dot(v_fail, v_gist):.6f}")

## 📦 Section 4: Curated 100-Task Failure-Intensive HumanEval Dataset Loader (Zero Overlap)

In [ ]:
import gzip
import json
import urllib.request
from datasets import load_dataset

# 1. First 50 failure-focused HumanEval Tasks (Completed)
FIRST_50_HUMANEVAL_IDS = [
    "HumanEval/0", "HumanEval/1", "HumanEval/2", "HumanEval/3", "HumanEval/4",
    "HumanEval/5", "HumanEval/6", "HumanEval/7", "HumanEval/8", "HumanEval/9",
    "HumanEval/10", "HumanEval/12", "HumanEval/13", "HumanEval/14", "HumanEval/16",
    "HumanEval/20", "HumanEval/21", "HumanEval/23", "HumanEval/25", "HumanEval/27",
    "HumanEval/29", "HumanEval/31", "HumanEval/32", "HumanEval/36", "HumanEval/38",
    "HumanEval/49", "HumanEval/50", "HumanEval/55", "HumanEval/65", "HumanEval/67",
    "HumanEval/70", "HumanEval/78", "HumanEval/84", "HumanEval/89", "HumanEval/93",
    "HumanEval/94", "HumanEval/96", "HumanEval/108", "HumanEval/110", "HumanEval/112",
    "HumanEval/115", "HumanEval/118", "HumanEval/122", "HumanEval/123", "HumanEval/127",
    "HumanEval/130", "HumanEval/131", "HumanEval/132", "HumanEval/140", "HumanEval/141"
]

# 2. Curated 50 hardest, failure-prone, edge-case intensive uncompleted tasks (Tasks 51 to 100)
SECOND_50_FAILURE_INTENSIVE_IDS = [
    "HumanEval/19",  # sort_numbers: text-to-number mapping
    "HumanEval/37",  # sort_even: even indices sorting preserving odd
    "HumanEval/64",  # vowels_count: conditional 'y' string boundary rule
    "HumanEval/74",  # total_match: list char count comparison
    "HumanEval/81",  # numerical_letter_grade: float boundary conditions
    "HumanEval/83",  # starts_one_ends: combinatorial counting
    "HumanEval/85",  # add: odd indices with even values
    "HumanEval/88",  # sort_array: parity-dependent sort direction
    "HumanEval/91",  # is_bored: multi-delimiter sentence start parsing
    "HumanEval/95",  # check_dict_case: mixed string key case validation
    "HumanEval/97",  # multiply: unit digit negative product
    "HumanEval/98",  # count_upper: uppercase vowels at even indices
    "HumanEval/99",  # closest_integer: round away from zero boundary
    "HumanEval/102", # choose_num: range max even number
    "HumanEval/104", # unique_digits: no even digits filter
    "HumanEval/109", # move_one_ball: circular sorted shift
    "HumanEval/111", # histogram: frequency dictionary with multi-mode
    "HumanEval/114", # minSubArraySum: min Kadane subarray sum
    "HumanEval/116", # sort_array: binary popcount sort
    "HumanEval/119", # match_parens: open/close paren concatenation
    "HumanEval/120", # maximum: top k elements sorted
    "HumanEval/121", # solution: sum of odd elements in even positions
    "HumanEval/124", # valid_date: multi-condition date regex & day limits
    "HumanEval/125", # split_words: multi-delimiter fallback split
    "HumanEval/126", # is_sorted: sorted + max 2 duplicates
    "HumanEval/128", # prod_signs: magnitude sum times sign product
    "HumanEval/129", # minPath: grid path minimization (DFS/BFS)
    "HumanEval/133", # sum_squares: ceil squared sum
    "HumanEval/134", # check_if_last_char_is_a_letter: word boundary check
    "HumanEval/135", # can_arrange: largest inverted index
    "HumanEval/136", # largest_smallest_integers: signed extremes
    "HumanEval/137", # compare_one: mixed type float/comma string comparisons
    "HumanEval/138", # is_equal_to_sum_even: 4 positive even sum
    "HumanEval/139", # special_factorial: product of factorials
    "HumanEval/142", # sum_squares: index 3/4 divisibility
    "HumanEval/143", # words_in_sentence: prime length word filtering
    "HumanEval/144", # simplify: fraction multiplication whole number check
    "HumanEval/145", # order_by_points: signed digit sum sorting
    "HumanEval/146", # specialFilter: >10 with both first and last digits odd
    "HumanEval/147", # get_max_triples: modular arithmetic (a+b+c)%3==0
    "HumanEval/148", # bf: planet order traversal
    "HumanEval/149", # sorted_list_sum: even length string sorting
    "HumanEval/150", # x_or_y: prime check branching
    "HumanEval/151", # double_the_difference: odd integer square sum
    "HumanEval/152", # compare: prediction error array
    "HumanEval/153", # Strongest_Extension: case strength difference
    "HumanEval/154", # cycpattern_check: circular substring matching
    "HumanEval/156", # int_to_mini_roman: integer to Roman numeral conversion
    "HumanEval/158", # find_max: max unique chars string with tiebreak
    "HumanEval/160"  # do_algebra: operand and operator evaluation
]

def categorize_task(task_id: str) -> str:
    num = int(task_id.split("/")[-1]) if "/" in task_id else 0
    if num in [0, 4, 13, 21, 25, 31, 32, 49, 55, 67, 83, 96, 97, 122, 123, 130, 131, 138, 139, 147, 150, 151, 160]:
        return "Math & Number Theory"
    elif num in [1, 6, 7, 10, 12, 14, 16, 19, 23, 27, 29, 38, 50, 64, 65, 89, 91, 93, 112, 118, 124, 125, 134, 137, 140, 141, 143, 148, 149, 153, 154, 156, 158]:
        return "String Parsing & Manipulation"
    elif num in [2, 3, 5, 8, 9, 20, 36, 37, 70, 74, 78, 84, 85, 88, 94, 95, 98, 99, 102, 104, 108, 109, 110, 111, 112, 114, 115, 116, 119, 120, 121, 126, 127, 128, 132, 133, 135, 136, 142, 145, 146, 152]:
        return "Array & List Operations"
    elif num in [129]:
        return "Dynamic Programming & Recursion"
    else:
        return "Logic & Validation Rules"

print("Loading Official HumanEval Dataset...")
raw_humaneval_dict = {}
try:
    ds = load_dataset("openai_humaneval", split="test")
    for item in ds:
        raw_humaneval_dict[item["task_id"]] = {
            "task_id": item["task_id"],
            "benchmark": "HumanEval",
            "prompt": item["prompt"],
            "entry_point": item["entry_point"],
            "test": item["test"],
            "category": categorize_task(item["task_id"])
        }
except Exception as e:
    print(f"Loading via fallback URL: {e}")
    url = "https://github.com/openai/human-eval/raw/master/data/HumanEval.jsonl.gz"
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp:
        with gzip.GzipFile(fileobj=resp) as gz:
            for line in gz:
                item = json.loads(line.decode('utf-8'))
                raw_humaneval_dict[item["task_id"]] = {
                    "task_id": item["task_id"],
                    "benchmark": "HumanEval",
                    "prompt": item["prompt"],
                    "entry_point": item["entry_point"],
                    "test": item["test"],
                    "category": categorize_task(item["task_id"])
                }

# Assemble exact 100 HumanEval tasks (50 Completed + 50 Hard Uncompleted)
selected_tasks = []
# 1. Add first 50
for tid in FIRST_50_HUMANEVAL_IDS:
    if tid in raw_humaneval_dict:
        selected_tasks.append(raw_humaneval_dict[tid])

# 2. Add second 50 (Zero Overlap guaranteed)
for tid in SECOND_50_FAILURE_INTENSIVE_IDS:
    if tid in raw_humaneval_dict:
        selected_tasks.append(raw_humaneval_dict[tid])

print(f"✅ Successfully assembled exactly {len(selected_tasks)} failure-intensive HumanEval benchmark tasks:")
print(f"  • Tasks 1 to 50: Historical Baseline Failure Tasks (Original failure set)")
print(f"  • Tasks 51 to 100: Curated Failure-Intensive Tasks (Zero Overlap)")

## 🧪 Section 5: Universal HumanEval Code Extractor & Isolated Test Execution Engine

In [ ]:
import ast
import subprocess
import tempfile
import textwrap

EXEC_TIMEOUT_S = 10  # Wall-clock limit per task for generated code + HumanEval assertions

def _trim(text: str) -> str:
    # Drop leading blank lines and trailing whitespace, but keep the first line's indentation
    return re.sub(r"^\s*\n", "", text).rstrip()

def extract_clean_code(generation: str, entry_point: str, docstring_prompt: str = "") -> str:
    code = generation
    if "</think>" in code:
        code = code.split("</think>")[-1]

    def_pattern = r"def\s+" + re.escape(entry_point) + r"\s*\("
    blocks = re.findall(r"```(?:python|py)?[ \t]*\n(.*?)(?:```|$)", code, re.DOTALL)
    if blocks:
        with_def = [b for b in blocks if re.search(def_pattern, b)]
        code = with_def[-1] if with_def else blocks[0]
    code = _trim(code)

    if not re.search(def_pattern, code):
        # Body-only completion: build_test_program re-attaches it under the original signature
        return code

    try:
        tree = ast.parse(code)
    except SyntaxError:
        # Unfenced answer with prose around the code: start at the first code-looking line,
        # then keep the longest prefix that parses and still contains the entry point
        first_code = re.search(r"^(import |from |def |class |@)", code, re.MULTILINE)
        code_lines = (code[first_code.start():] if first_code else code).splitlines()
        tree = None
        for end in range(len(code_lines), 0, -1):
            candidate = "\n".join(code_lines[:end])
            if not re.search(def_pattern, candidate):
                break
            try:
                tree = ast.parse(candidate)
                code = candidate
                break
            except SyntaxError:
                continue
        if tree is None:
            return code  # The test run reports the SyntaxError

    # Keep defs, classes, imports and constant assignments; drop top-level demo code
    # (print(...), asserts, `if __name__ == "__main__":`, `x = input()`, `result = entry_point(...)`)
    lines = code.splitlines()
    kept = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Import, ast.ImportFrom)):
            keep = True
        elif isinstance(node, (ast.Assign, ast.AnnAssign)):
            called = {n.func.id for n in ast.walk(node) if isinstance(n, ast.Call) and isinstance(n.func, ast.Name)}
            keep = not (called & {"input", "print", entry_point})
        else:
            keep = False
        if keep:
            first_line = min([node.lineno] + [d.lineno for d in getattr(node, "decorator_list", [])])
            kept.append("\n".join(lines[first_line - 1:node.end_lineno]))
    return "\n\n".join(kept)

HARNESS_PREAMBLE = "import math\nimport re\nimport sys\nfrom typing import List, Tuple, Dict, Any, Optional, Union\n\n"

def build_test_program(code: str, test_script: str, entry_point: str, docstring_prompt: str = "") -> str:
    has_def = bool(re.search(r"def\s+" + re.escape(entry_point) + r"\s*\(", code))
    if not has_def and docstring_prompt:
        body = textwrap.indent(textwrap.dedent(code), "    ")
        code = docstring_prompt.rstrip() + "\n" + body
    # HumanEval's test field only defines check(candidate); it must be called explicitly
    return f"{HARNESS_PREAMBLE}{code}\n\n{test_script}\n\ncheck({entry_point})\n"

def execute_unit_test(code: str, test_script: str, entry_point: str, docstring_prompt: str = "", timeout: int = EXEC_TIMEOUT_S) -> Tuple[bool, str, str]:
    program = build_test_program(code, test_script, entry_point, docstring_prompt)
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False, encoding="utf-8") as f_prog:
        f_prog.write(program)
        program_path = f_prog.name
    try:
        proc = subprocess.run(
            [sys.executable, "-I", program_path],
            capture_output=True, text=True, timeout=timeout, stdin=subprocess.DEVNULL
        )
    except subprocess.TimeoutExpired:
        return False, "Timeout", f"Timeout: exceeded {timeout}s (infinite loop or blocking call)."
    finally:
        os.remove(program_path)

    if proc.returncode == 0:
        return True, "SUCCESS", "Passed all HumanEval assertions."
    stderr_lines = [line for line in proc.stderr.strip().splitlines() if line.strip()]
    last_line = stderr_lines[-1].strip() if stderr_lines else f"Exit code {proc.returncode}"
    err_name = last_line.split(":")[0].strip()
    if not err_name.isidentifier():
        err_name = "RuntimeError"
    if err_name == "AssertionError":
        return False, err_name, "AssertionError (Logic Failure): Output did not match assertion."
    return False, err_name, last_line


## 🧪 Section 6A: Phase A: Vanilla Baseline on All 100 Tasks (Resumable)

In [ ]:
ALPHA = 0.08
BETA = 0.015
TAU = round(2.0 / math.sqrt(D_MODEL), 3)  # Theoretical 2-sigma threshold for a random unit-vector cosine (d_model=2560 -> 0.04)
TOKEN_BUDGET = 1024

COMPARISON_VARIANTS = ["neg_prompt", "gated_steered", "dual_qwen"]
variants = ["vanilla"] + COMPARISON_VARIANTS
VANILLA_CHECKPOINT = "qwen3_phase_a_vanilla_checkpoint.json"
COMPARISON_CHECKPOINT = "qwen3_phase_b_failures_checkpoint.json"
CHUNK_SIZE = 20  # Tasks evaluated per cell run; re-run the cell to continue with the next chunk

labels_map = {
    "vanilla": "1. Vanilla Baseline",
    "neg_prompt": "2. Negative System Prompting",
    "gated_steered": f"3. Adaptive RepGated (tau={TAU})",
    "dual_qwen": "4. Dual Push-Pull Qwen"
}
short_names = {"vanilla": "Vanilla", "neg_prompt": "NegPrompt", "gated_steered": "RepGated", "dual_qwen": "DualQwen"}

def load_checkpoint(path: str) -> Dict[str, Any]:
    # Resume only runs produced with identical model / decoding settings
    if not os.path.exists(path):
        return {}
    with open(path, "r", encoding="utf-8") as f_chk:
        chk = json.load(f_chk)
    if (chk.get("model_name"), chk.get("enable_thinking"), chk.get("token_budget")) != (MODEL_NAME, ENABLE_THINKING, TOKEN_BUDGET):
        print(f"⚠️ Ignoring {path}: produced with different model/decoding settings.")
        return {}
    return chk["results"]

def save_checkpoint(path: str, results: Dict[str, Any]):
    with open(path, "w", encoding="utf-8") as f_chk:
        json.dump({
            "model_name": MODEL_NAME,
            "enable_thinking": ENABLE_THINKING,
            "token_budget": TOKEN_BUDGET,
            "results": results
        }, f_chk)

def generate_plain(prompt: str) -> Tuple[str, Dict[str, Any]]:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=TOKEN_BUDGET, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    gen_tokens = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True), {"total_tokens": len(gen_tokens)}

def run_variant(variant: str, task: Dict[str, Any]) -> Dict[str, Any]:
    raw_prompt = task["prompt"]
    if variant == "neg_prompt":
        prompt = format_negative_system_prompt(tokenizer, raw_prompt)
    else:
        prompt = format_chatml_prompt(tokenizer, raw_prompt)

    t0 = time.time()
    if variant in ("vanilla", "neg_prompt"):
        txt, stat = generate_plain(prompt)
    elif variant == "gated_steered":
        txt, stat = generator_gated.generate_gated(prompt, v_fail=v_fail, target_layer=TARGET_LAYER, alpha=ALPHA, tau=TAU, max_new_tokens=TOKEN_BUDGET)
    elif variant == "dual_qwen":
        txt, stat = generator_dual.generate_dual_steered(prompt, v_fail=v_fail, v_gist=v_gist, target_layer=TARGET_LAYER, alpha=ALPHA, beta=BETA, max_new_tokens=TOKEN_BUDGET)
    else:
        raise ValueError(f"Unknown variant: {variant}")
    elapsed = time.time() - t0

    code = extract_clean_code(txt, task["entry_point"], raw_prompt)
    passed, err, trace = execute_unit_test(code, task["test"], task["entry_point"], raw_prompt)
    diag = entropy_evaluator.compute_token_diagnostics(prompt, txt)
    n_tokens = stat["total_tokens"]
    return {
        "pass": passed, "err": err, "trace": trace, "txt": txt, "code": code,
        "time": elapsed, "tokens": n_tokens, "truncated": n_tokens >= TOKEN_BUDGET,
        "gating_ratio": stat.get("gating_ratio", 100.0 if variant == "dual_qwen" else 0.0),
        "entropy": diag["mean_entropy"], "r_rep": diag["r_rep"],
        "entropy_traj": diag["entropy_trajectory"],
        "s_fail_traj": stat.get("s_fail_history", []),
        "s_gist_traj": stat.get("s_gist_history", [])
    }

vanilla_logs = load_checkpoint(VANILLA_CHECKPOINT)
pending_tasks = [t for t in selected_tasks if t["task_id"] not in vanilla_logs]
chunk = pending_tasks[:CHUNK_SIZE]
print(f"🚀 Phase A: {len(vanilla_logs)}/{len(selected_tasks)} tasks done. Evaluating the next {len(chunk)} (thinking={ENABLE_THINKING}, budget={TOKEN_BUDGET} tokens)...")

for task in chunk:
    task_id = task["task_id"]
    idx = selected_tasks.index(task) + 1
    vanilla_logs[task_id] = {
        "task_id": task_id, "benchmark": task.get("benchmark", "HumanEval"),
        "entry_point": task["entry_point"], "category": task["category"],
        "outputs": {"vanilla": run_variant("vanilla", task)}
    }
    save_checkpoint(VANILLA_CHECKPOINT, vanilla_logs)

    out_v = vanilla_logs[task_id]["outputs"]["vanilla"]
    sym_v = "✅" if out_v["pass"] else "❌"
    trunc_tag = " ✂️ TRUNCATED" if out_v["truncated"] else ""
    print(f"[{idx:03d}/{len(selected_tasks):03d}] {task_id:<15} ({task['entry_point']:<32}) | Vanilla: {sym_v} {out_v['err']:<15} | {out_v['tokens']:>4} tok{trunc_tag}", flush=True)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

n_tasks = len(selected_tasks)
remaining = n_tasks - len(vanilla_logs)
n_done_pass = sum(1 for log in vanilla_logs.values() if log["outputs"]["vanilla"]["pass"])
if remaining > 0:
    print(f"\n⏸️ Phase A progress: {len(vanilla_logs)}/{n_tasks} tasks done ({n_done_pass} passed so far), {remaining} remaining.")
    print(f"   Re-run this cell to evaluate the next {min(CHUNK_SIZE, remaining)} tasks.")
else:
    failed_task_ids = [t["task_id"] for t in selected_tasks if not vanilla_logs[t["task_id"]]["outputs"]["vanilla"]["pass"]]
    n_vanilla_pass = n_tasks - len(failed_task_ids)

    failure_breakdown = {}
    for tid in failed_task_ids:
        err = vanilla_logs[tid]["outputs"]["vanilla"]["err"]
        failure_breakdown[err] = failure_breakdown.get(err, 0) + 1
    n_truncated = sum(1 for tid in failed_task_ids if vanilla_logs[tid]["outputs"]["vanilla"]["truncated"])

    print("\n" + "=" * 115)
    print(f"PHASE A COMPLETE | Vanilla Pass@1: {n_vanilla_pass}/{n_tasks} ({100 * n_vanilla_pass / n_tasks:.1f}%)")
    print(f"Vanilla failures carried to Phase B: {len(failed_task_ids)} | by error: {failure_breakdown} | truncated: {n_truncated}")
    print("=" * 115)
    for tid in failed_task_ids:
        o = vanilla_logs[tid]["outputs"]["vanilla"]
        print(f"  ❌ {tid:<15} {vanilla_logs[tid]['entry_point']:<32} {o['err']:<15} {o['trace'][:70]}")


## 🎯 Section 6B: Phase B: Re-run Vanilla Failures with NegPrompt, RepGated & Dual Qwen (Resumable)

In [ ]:
assert len(vanilla_logs) == len(selected_tasks), "Phase A is not finished: re-run the Phase A cell until all 100 tasks are done."

failed_set = set(failed_task_ids)
failed_tasks = [t for t in selected_tasks if t["task_id"] in failed_set]
comparison_logs = load_checkpoint(COMPARISON_CHECKPOINT)
pending_tasks = [
    t for t in failed_tasks
    if not all(v in comparison_logs.get(t["task_id"], {}).get("outputs", {}) for v in COMPARISON_VARIANTS)
]
chunk = pending_tasks[:CHUNK_SIZE]

print(f"🎯 Phase B: {len(failed_tasks) - len(pending_tasks)}/{len(failed_tasks)} vanilla failures done. "
      f"Re-running the next {len(chunk)} with {', '.join(short_names[v] for v in COMPARISON_VARIANTS)}...")
for task in chunk:
    task_id = task["task_id"]
    idx = failed_tasks.index(task) + 1
    entry = comparison_logs.setdefault(task_id, {"outputs": {}})
    for v_name in COMPARISON_VARIANTS:
        if v_name not in entry["outputs"]:
            entry["outputs"][v_name] = run_variant(v_name, task)
            save_checkpoint(COMPARISON_CHECKPOINT, comparison_logs)

    row = " | ".join(f"{short_names[v]}: {'✅' if entry['outputs'][v]['pass'] else '❌'}" for v in COMPARISON_VARIANTS)
    print(f"[{idx:03d}/{len(failed_tasks):03d}] {task_id:<15} ({task['entry_point']:<32}) | Vanilla: ❌ | {row}", flush=True)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

remaining = len(pending_tasks) - len(chunk)
phase_b_complete = remaining == 0
if not phase_b_complete:
    print(f"\n⏸️ Phase B progress: {len(failed_tasks) - remaining}/{len(failed_tasks)} failures done, {remaining} remaining.")
    print(f"   Re-run this cell to evaluate the next {min(CHUNK_SIZE, remaining)} tasks.")
else:
    # Merge vanilla + comparison outputs for every vanilla-failed task
    benchmark_logs = []
    for tid in failed_task_ids:
        merged = dict(vanilla_logs[tid])
        merged["outputs"] = {**vanilla_logs[tid]["outputs"], **comparison_logs[tid]["outputs"]}
        benchmark_logs.append(merged)

    print("\n" + "=" * 115)
    print(f"      PHASE B COMPLETE: {len(benchmark_logs)} vanilla failures re-evaluated with {len(COMPARISON_VARIANTS)} methods")
    print("=" * 115)


## 📈 Section 7: Failure-Recovery Table & Statistical Dashboard

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def compute_wilson_ci(k: int, n: int, confidence: float = 0.95) -> Tuple[float, float]:
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    spread = z * math.sqrt((p * (1 - p) + z**2 / (4 * n)) / n)
    lower = max(0.0, (centre - spread) / denominator) * 100
    upper = min(1.0, (centre + spread) / denominator) * 100
    return lower, upper

def compute_mcnemar_exact(b_baseline_only: int, c_steered_only: int) -> float:
    # Exact two-sided binomial test on the discordant pairs
    total_discordant = b_baseline_only + c_steered_only
    if total_discordant == 0:
        return 1.0
    return float(stats.binomtest(c_steered_only, total_discordant, 0.5).pvalue)

assert globals().get("phase_b_complete"), "Phase B is not finished: re-run the Phase B cell until all failures are done."

colors_dict = {"vanilla": "#4A5568", "neg_prompt": "#ED8936", "gated_steered": "#805AD5", "dual_qwen": "#38A169"}
n_fail = len(benchmark_logs)

if n_fail == 0:
    print(f"Vanilla passed all {n_tasks} tasks: there are no failures to recover.")
else:
    summary_table_data = []
    for v in variants:
        outs = [t["outputs"][v] for t in benchmark_logs]
        k = sum(1 for o in outs if o["pass"])
        total_time = sum(o["time"] for o in outs)
        total_tok = sum(o["tokens"] for o in outs)
        summary_table_data.append({
            "key": v, "k": k,
            "rate": 100 * k / n_fail,
            "ci": compute_wilson_ci(k, n_fail),
            # Vanilla failed every task here, so each recovery is a steered-only discordant pair
            "p": None if v == "vanilla" else compute_mcnemar_exact(0, k),
            "projected": 100 * (n_vanilla_pass + k) / n_tasks,
            "entropy": float(np.mean([o["entropy"] for o in outs])),
            "r_rep": float(np.mean([o["r_rep"] for o in outs])),
            "trunc": 100 * float(np.mean([o["truncated"] for o in outs])),
            "throughput": total_tok / total_time if total_time > 0 else 0.0
        })

    print("\n" + "=" * 150)
    print(f"FAILURE-RECOVERY RESULTS on {n_fail} vanilla-failed tasks (vanilla Pass@1 on all {n_tasks}: {n_vanilla_pass}/{n_tasks})")
    print("=" * 150)
    print(f"{'Method':<34} | {'Recovered':<10} | {'Rate':<7} | {'95% Wilson CI':<16} | {'McNemar p':<12} | {'Proj. Pass@1':<12} | {'H(X) bits':<9} | {'R_rep':<7} | {'Trunc %':<7} | {'tok/s':<7}")
    print("-" * 150)
    for d in summary_table_data:
        p_str = "n/a" if d["p"] is None else (f"{d['p']:.4f}" + (" ✅" if d["p"] < 0.05 else ""))
        print(
            f"{labels_map[d['key']]:<34} | {d['k']:>3}/{n_fail:<6} | {d['rate']:5.1f}% | "
            f"[{d['ci'][0]:5.1f}, {d['ci'][1]:5.1f}]% | {p_str:<12} | {d['projected']:6.1f}%      | "
            f"{d['entropy']:9.3f} | {d['r_rep']:7.4f} | {d['trunc']:6.1f}% | {d['throughput']:7.2f}"
        )
    print("=" * 150)
    print(f"Projected Pass@1 = (vanilla passes + recovered) / {n_tasks}. It is an upper bound: the methods were not re-run on the")
    print(f"{n_vanilla_pass} tasks vanilla already solved, so regressions on those tasks are not measured.")

    comp = summary_table_data[1:]
    comp_names = [short_names[d["key"]] for d in comp]
    comp_colors = [colors_dict[d["key"]] for d in comp]
    all_names = [short_names[d["key"]] for d in summary_table_data]
    all_colors = [colors_dict[d["key"]] for d in summary_table_data]

    fig, axs = plt.subplots(2, 3, figsize=(20, 11))
    fig.suptitle(f"Qwen3-4B: Failure-Recovery Steering on {n_fail} Vanilla-Failed HumanEval Tasks (of {n_tasks})", fontsize=16, fontweight="bold", y=0.98)

    def label_bars(ax, bars, fmt):
        pad = 0.01 * max((bar.get_height() for bar in bars), default=0.0)
        for bar in bars:
            yval = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, yval + pad, fmt.format(yval), ha="center", va="bottom", fontweight="bold", fontsize=10)

    # Panel 1: Recovery rate with 95% Wilson CIs
    rates = [d["rate"] for d in comp]
    yerrs = [[d["rate"] - d["ci"][0] for d in comp], [d["ci"][1] - d["rate"] for d in comp]]
    bars1 = axs[0, 0].bar(comp_names, rates, color=comp_colors, yerr=yerrs, capsize=5, edgecolor="black", alpha=0.9)
    axs[0, 0].set_title("1. Recovery Rate on Vanilla Failures (95% Wilson CIs)", fontweight="bold")
    axs[0, 0].set_ylabel("Recovered (%)")
    axs[0, 0].set_ylim(0, 100)
    label_bars(axs[0, 0], bars1, "{:.1f}%")

    # Panel 2: Shannon entropy on the failure subset
    bars2 = axs[0, 1].bar(all_names, [d["entropy"] for d in summary_table_data], color=all_colors, edgecolor="black", alpha=0.9)
    axs[0, 1].set_title("2. Shannon Entropy H(X) on Failure Subset", fontweight="bold")
    axs[0, 1].set_ylabel("Entropy (bits)")
    label_bars(axs[0, 1], bars2, "{:.3f}")

    # Panel 3: 3-gram repetition ratio
    bars3 = axs[0, 2].bar(all_names, [d["r_rep"] for d in summary_table_data], color=all_colors, edgecolor="black", alpha=0.9)
    axs[0, 2].set_title("3. 3-Gram Repetition Ratio R_rep", fontweight="bold")
    axs[0, 2].set_ylabel("Repetition Ratio")
    label_bars(axs[0, 2], bars3, "{:.4f}")

    # Panel 4: Error taxonomy
    all_errs = sorted({t["outputs"][v]["err"] for t in benchmark_logs for v in variants} - {"SUCCESS"})
    category_names = ["SUCCESS"] + all_errs
    error_palette = ["#E53E3E", "#D69E2E", "#3182CE", "#805AD5", "#DD6B20", "#718096", "#D53F8C", "#319795"]
    bottoms = np.zeros(len(variants))
    for c_idx, cat in enumerate(category_names):
        vals = np.array([sum(1 for t in benchmark_logs if t["outputs"][v]["err"] == cat) for v in variants])
        color = "#38A169" if cat == "SUCCESS" else error_palette[(c_idx - 1) % len(error_palette)]
        axs[1, 0].bar(all_names, vals, bottom=bottoms, label=cat, color=color, edgecolor="white")
        bottoms += vals
    axs[1, 0].set_title("4. Execution Error Taxonomy (Failure Subset)", fontweight="bold")
    axs[1, 0].set_ylabel("Task Count")
    axs[1, 0].legend(loc="upper right", fontsize=8)

    # Panel 5: Throughput
    bars5 = axs[1, 1].bar(all_names, [d["throughput"] for d in summary_table_data], color=all_colors, edgecolor="black", alpha=0.9)
    axs[1, 1].set_title("5. Generation Throughput (tokens/sec) on Tesla T4", fontweight="bold")
    axs[1, 1].set_ylabel("Tokens / Second")
    label_bars(axs[1, 1], bars5, "{:.1f}")

    # Panel 6: Projected overall Pass@1
    bars6 = axs[1, 2].bar(all_names, [d["projected"] for d in summary_table_data], color=all_colors, edgecolor="black", alpha=0.9)
    axs[1, 2].set_title(f"6. Projected Pass@1 on All {n_tasks} Tasks (upper bound)", fontweight="bold")
    axs[1, 2].set_ylabel("Pass@1 (%)")
    axs[1, 2].set_ylim(0, 105)
    label_bars(axs[1, 2], bars6, "{:.1f}%")

    for ax in axs.flat:
        ax.tick_params(axis="x", labelsize=9)
    plt.tight_layout()
    plt.show()

    # Per-task recovery heatmap
    matrix = np.array([[1 if t["outputs"][v]["pass"] else 0 for v in COMPARISON_VARIANTS] for t in benchmark_logs])
    fig_h, ax_h = plt.subplots(figsize=(7, max(3, 0.32 * n_fail)))
    ax_h.imshow(matrix, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax_h.set_xticks(range(len(COMPARISON_VARIANTS)))
    ax_h.set_xticklabels([short_names[v] for v in COMPARISON_VARIANTS])
    ax_h.set_yticks(range(n_fail))
    ax_h.set_yticklabels([f"{t['task_id']} ({t['entry_point']})" for t in benchmark_logs], fontsize=8)
    ax_h.set_title("Per-Task Recovery (green = recovered)", fontweight="bold")
    plt.tight_layout()
    plt.show()


## 🔍 Section 8: Interactive Task-by-Task Trajectory Comparator & Quality Inspector

In [ ]:
def plot_task_trajectory_comparison(task_id: str):
    found = [entry for entry in benchmark_logs if entry["task_id"] == task_id]
    if not found:
        print(f"Task {task_id} is not a vanilla failure (only Phase B tasks have all methods).")
        return

    t_log = found[0]
    print("=" * 100)
    print(f"           STEP-BY-STEP LATENT TRAJECTORY COMPARISON: {task_id} ({t_log['entry_point']})")
    print(f"           Category: {t_log['category']}")
    print("=" * 100)
    for v_name in variants:
        v_out = t_log["outputs"][v_name]
        status_sym = "✅ PASS" if v_out["pass"] else "❌ FAIL"
        print(f"{labels_map[v_name]:<34} {status_sym}  {v_out['err']:<15} {v_out['tokens']:>4} tok  {v_out['trace'][:60]}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    for v_name in variants:
        v_out = t_log["outputs"][v_name]
        status_sym = "PASS" if v_out["pass"] else "FAIL"
        ent_traj = v_out.get("entropy_traj", [])
        if ent_traj:
            ax1.plot(ent_traj[:150], label=f"{labels_map[v_name]} ({status_sym})", color=colors_dict[v_name], linewidth=1.8)

        s_fail_traj = v_out.get("s_fail_traj", [])
        if s_fail_traj:
            ax2.plot(s_fail_traj[:150], label=f"{labels_map[v_name]}", color=colors_dict[v_name], linewidth=1.8)

    ax1.set_title(f"Per-Token Shannon Entropy Trajectory: {task_id}", fontweight="bold")
    ax1.set_xlabel("Token Decoding Step (t)")
    ax1.set_ylabel("Entropy H(p_t) (bits)")
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=8)

    ax2.set_title(f"Per-Token Latent Failure Cosine Projection: {task_id}", fontweight="bold")
    ax2.set_xlabel("Token Decoding Step (t)")
    ax2.set_ylabel("Cosine Margin S_{t, fail}")
    ax2.axhline(y=TAU, color="red", linestyle="--", alpha=0.7, label=f"Gating Threshold tau={TAU}")
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

# Example: inspect the first vanilla failure recovered by Dual Qwen (or the first failure if none recovered)
if benchmark_logs:
    example_id = next((t["task_id"] for t in benchmark_logs if t["outputs"]["dual_qwen"]["pass"]), benchmark_logs[0]["task_id"])
    plot_task_trajectory_comparison(example_id)
